# ap2101_extract_summed_eeg_data

## Exporting Data for a Single Organization ID totaled fo Meter Point ID's

### Notebook Structure

1. **Imports & Definitions**  
    Initialization of required libraries, configuration settings, and helper functions for database access and file handling.
    
2. **Exporting Data From Database**  
    Retrieval of all relevant data for the specified **OrgID** and prepare the date
    
3. **Save as CSV**  
    All retrieved datasets are exported to CSV files.

### 01 Imports & Definitions

In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from datetime import datetime
from modules.data_loading import load_energy_community_data, load_all_metering_points_in_energy_community_data, get_postgres_engine

In [ ]:
org_id = 10
time_start = datetime(2025, 1, 1)
time_end = datetime(2025, 9, 30)

path_to_local_data = "../../local_data/"
date = datetime.now()   # aktuelles Datum
output_filename = f"{path_to_local_data}raw/summed_eeg_orgid-{org_id}-{date}"


### 02 Exporting Data From Database

In [ ]:
load_dotenv()
ssh_host = os.getenv("SSH_HOST")
ssh_port = int(os.getenv("SSH_PORT"))
ssh_user = os.getenv("SSH_USER")
ssh_pw = os.getenv("SSH_PASSWORD")

postgres_server_ip = os.getenv("POSTGRES_SERVER_IP")
postgres_port = int(os.getenv("POSTGRES_PORT"))

In [ ]:
with SSHTunnelForwarder(
    (ssh_host, ssh_port),
    ssh_username=ssh_user,
    ssh_password=ssh_pw,
    remote_bind_address=(ssh_host, postgres_port),
    local_bind_address=(postgres_server_ip, postgres_port)
) as tunnel:
    df=load_energy_community_data(org_id=org_id, time_start=time_start, time_end=time_end, sql_engine=get_postgres_engine())
    print(df.head())

### 03 Export to csv

In [ ]:
df.to_csv(output_filename+".csv", index=False)

In [ ]:
descriptions = {
    "Data extraction date": date,
    "org_id": "Organisation ID",
    "time": "Measurement timestamp",
    "metering_points_cnt": "Number of metering points in the REC",
    "consumer_count": "Number of consumers in the REC",
    "generator_count": "Number of producers in the REC",
    "sum_wt_meas_cons": "Sum of measured consumption weighted by participation factor",
    "sum_comm_pot": "Sum of community potential",
    "sum_comm_cov": "Sum of community coverage",
    "sum_wt_meas_gen": "Sum of measured generation weighted by participation factor",
    "sum_wt_surp_gen": "Sum of residual surplus weighted by participation factor"
}

# DataFrame with columns
df = pd.DataFrame(columns=descriptions.keys())

# create CSV
meta_df = pd.DataFrame({
    "column_name": df.columns,
    "description": df.columns.map(descriptions)
})

meta_df.to_csv(output_filename+"_metadata.csv", index=False)